In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import snapshot_download
from datasets import load_dataset, concatenate_datasets

/opt/miniconda3/envs/py312/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Downloading the Tokenizer and Vocab files

In [2]:
# Define the revision we want to download
rev = "stage1-step928646-tokens3896B"

# Specify the absolute path
# model_dir = f"/Users/nirajh/Code/Language_adaptation/olmo2-7b-{rev.split('-')[0]}"

In [3]:
# snapshot_download(
#     repo_id="allenai/OLMo-2-1124-7B",
#     revision=rev,
#     local_dir=model_dir,
#     allow_patterns=[
#         "*tokenizer*",
#         "*vocab*",
#         "*.json",
#         "*.txt",
#         "tokenizer_config.json",
#     ],
#     ignore_patterns=["*.safetensors", "*.bin", "*.h5", "*.ot"],
# )

In [5]:
# model_dir = f"/Users/nirajh/Code/Language_adaptation/olmo2-7b-{rev.split('-')[0]}-full"

# snapshot_download(
#     repo_id="allenai/OLMo-2-1124-7B",
#     revision=rev,
#     local_dir=model_dir
# )


In [ ]:
model = AutoModelForCausalLM.from_pretrained(model_dir, dtype="bfloat16")

In [ ]:
tok = AutoTokenizer.from_pretrained(model_dir)

In [ ]:
ip_text = "hello, how are you?"

tokens = tok.tokenize(ip_text)
token_ids = tok.encode(ip_text)

print("Tokens: ",tokens)
print("Token IDs: ",token_ids)

Tokens:  ['hello', ',', 'Ġhow', 'Ġare', 'Ġyou', '?']
Token IDs:  [15339, 11, 1268, 527, 499, 30]


In [ ]:
ip_text = "ನಮಸ್ಕಾರ"

tokens = tok.tokenize(ip_text)
token_ids = tok.encode(ip_text)

print("Tokens: ",tokens)
print("Token IDs: ",token_ids)


Tokens:  ['à²', '¨', 'à²', '®', 'à²', '¸', 'à³', 'į', 'à²', 'ķ', 'à²', '¾', 'à²', '°']
Token IDs:  [34656, 101, 34656, 106, 34656, 116, 56990, 235, 34656, 243, 34656, 122, 34656, 108]


### Downloading the Kannada dataset

In [ ]:
dataset_cache_dir = "/Users/nirajh/Code/Language_adaptation/sangraha/"

In [ ]:
# Load the dataset and save it to the specified path
dataset = load_dataset(
    "ai4bharat/sangraha",
    data_dir="verified/kan",
    cache_dir=dataset_cache_dir
)

Generating train split: 3632345 examples [00:20, 179484.97 examples/s]


In [ ]:
# Keeping the 2000 elements that were originally in the test set
split = dataset["train"].train_test_split(test_size=2000, seed=42)
kn_train, kn_eval = split["train"], split["test"]

In [ ]:
# Add 9,000 new examples to the existing test set
additional = kn_train.train_test_split(
    test_size=362000,
    seed=42
)

kn_eval = concatenate_datasets([
    kn_eval,
    additional["test"]
])

kn_train = additional["train"]

print("Train set:\n",kn_train)
print("Test set:\n",kn_eval)

In [ ]:
kn_eval.save_to_disk("/Users/nirajh/Code/Language_adaptation/sangraha/train_test")

In [ ]:
kn_train.save_to_disk("/Users/nirajh/Code/Language_adaptation/sangraha/train_split")